# Data Preprocessing and Feature Engineering

Prepare the housing data using the saved train/test split and decisions documented during EDA. Imputation, encoding, and scaling will be fitted within training folds to prevent data leakage. The test set will remain reserved for final evaluation.

## 1. Import Required Libraries

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load the Saved Training Split

Reconstruct the training dataset using the previously saved row indices. Preserve the original row order of the raw dataset so that the saved indices identify the correct records.

In [4]:
project_root = Path.cwd().parent

df = pd.read_csv(project_root / "data/raw/melb_data.csv")
train_indices = pd.read_csv(
    project_root / "data/processed/train_indices.csv"
)["row_index"]

train_df = df.loc[train_indices].copy()

train_df.shape

(10864, 21)

### 3.1 Treat Zero Building Area as Missing

EDA identified 15 zero building-area values. Treat these as unavailable measurements while retaining the property records.

In [5]:
clean_train = train_df.copy()

clean_train["BuildingArea"] = clean_train["BuildingArea"].replace(0, np.nan)

clean_train["BuildingArea"].isna().sum()

np.int64(5144)

### 3.2 Treat Zero Land Size in Houses as Missing

Treat zero `Landsize` values for houses as unavailable information. Retain zeros for units and townhouses because shared-land recording conventions may apply.

In [6]:
mask = (clean_train["Type"] == "h") & (clean_train["Landsize"] == 0)

clean_train.loc[mask, "Landsize"] = np.nan

clean_train["Landsize"].isna().sum()

np.int64(138)